# MeanFlow `mf_loss` deviation from 1.0 — full val split (GPU)

Matches **training** (`experiment=downscaling_LMM_res_2mT_pretrain`):
- `MeanFlowPaperCore`, `crop_size=512`, `batch_size=25`, `nn_lowres=false`
- Same forward as `validation_step`: `create_graph=False`, `adaptive_l2_loss`

Scans the **validation split** (70/15/15 seed 42, same as training). Stores per-batch `mf_loss` in **float32** (training) and recomputes loss on **`error.double()`** for **float64** sensitivity.

Submit on cluster: `bash notebooks/Submit_mf_loss_probe.sh` (1× H100 MIG 3g.40gb).

**Why slower than `models_inference.ipynb`?** Inference times **165 test timesteps** (`nr_timesteps=165`). This probe scans the **full val split** (~15% of ~184k rows → **~27k crops**, **~1105 batches** at `batch_size=25`). Each batch also runs the **training loss path** (context encoder + VAE encode + **JVP** ≈ 2× `mf_unet`), not the lightweight 1–5 step sampler.

In [ ]:
import os
import sys
from pathlib import Path

import torch
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import open_dict

_REPO = Path.cwd()
if (_REPO / "src").is_dir():
    pass
elif (_REPO.parent / "src").is_dir():
    _REPO = _REPO.parent
    os.chdir(_REPO)
else:
    raise RuntimeError("Run from repo root or notebooks/")

sys.path.insert(0, str(_REPO))
os.environ.setdefault("PROJECT_ROOT", str(_REPO))

from scripts.mf_loss_probe_loop import run_val_mf_loss_probe, save_probe_results

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MAX_VAL_BATCHES = int(os.environ["MAX_VAL_BATCHES"]) if os.environ.get("MAX_VAL_BATCHES") else None

print("repo:", _REPO)
print("device:", DEVICE)
print("MAX_VAL_BATCHES:", MAX_VAL_BATCHES)

In [ ]:
def _dir_slash(p: str) -> str:
    return p if p.endswith(("/", "\\")) else p + "/"


def patch_paths(cfg) -> None:
    with open_dict(cfg.paths):
        cfg.paths.output_dir = str(_REPO / "logs" / "mf_loss_probe")
        cfg.paths.work_dir = str(_REPO)


data_dir = _dir_slash(os.environ.get("LDM_DATA_ROOT", str(_REPO / "LDM-downscaling" / "full_Dataset")))
metadata_dir = data_dir
if not os.path.isfile(data_dir + "metadata.csv"):
    metadata_dir = _dir_slash(str(Path(data_dir).parent))

cfg_dir = str(_REPO / "configs")
with initialize_config_dir(version_base="1.3", config_dir=cfg_dir):
    cfg = compose(
        config_name="train",
        overrides=[
            "experiment=downscaling_LMM_res_2mT_pretrain",
            f"paths.data_dir={data_dir}",
            f"paths.pretrained_models_dir={_REPO}/pretrained_models/",
        ],
    )
patch_paths(cfg)

with open_dict(cfg.data):
    cfg.data.data_dir = data_dir
    cfg.data.metadata_dir = metadata_dir

print("=== config (training-aligned) ===")
print("use_meanflow_paper_core:", cfg.model.use_meanflow_paper_core)
print("crop_size:", cfg.data.crop_size)
print("batch_size:", cfg.data.batch_size)
print("nn_lowres:", cfg.data.nn_lowres)
print("ckpt_path:", cfg.get("ckpt_path"))

lmm = instantiate(cfg.model).to(DEVICE)
ckpt_path = cfg.get("ckpt_path")
if ckpt_path and Path(ckpt_path).is_file():
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    lmm.load_state_dict(ckpt["state_dict"], strict=False)
    print("loaded:", ckpt_path)
else:
    raise FileNotFoundError(f"Missing LMM ckpt: {ckpt_path}")
lmm.eval()

dm = instantiate(cfg.data)
dm.prepare_data()
dm.setup("fit")
val_loader = dm.val_dataloader()
print("val batches:", len(val_loader), "(capped by MAX_VAL_BATCHES if set)")

In [ ]:
results = run_val_mf_loss_probe(
    lmm,
    val_loader,
    DEVICE,
    max_batches=MAX_VAL_BATCHES,
    show_progress=True,
)
summary = results["summary"]
print("\n=== mf_loss - 1 (float32, training path) ===")
for k, v in summary["mf_minus_1_f32"].items():
    print(f"  {k}: {v}")
print("\n=== mf_loss - 1 (float64 on error tensor) ===")
for k, v in summary["mf_minus_1_f64"].items():
    print(f"  {k}: {v}")
print("\nmax |mf_f64 - mf_f32| per batch:", summary["max_abs_diff_mf_f64_minus_f32"])

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

payload = {
    "config": {
        "experiment": "downscaling_LMM_res_2mT_pretrain",
        "crop_size": int(cfg.data.crop_size),
        "batch_size": int(cfg.data.batch_size),
        "ckpt_path": str(ckpt_path),
        "max_val_batches": MAX_VAL_BATCHES,
    },
    "summary": summary,
}
out_json = _REPO / "outputs" / "mf_loss_probe_stats.json"
save_probe_results(out_json, payload)

d32 = np.asarray(results["dev_f32"], dtype=np.float64)
d64 = np.asarray(results["dev_f64"], dtype=np.float64)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(d32, bins=80, color="steelblue", edgecolor="none")
axes[0].axvline(0, color="crimson", lw=1)
axes[0].set_title("mf_loss - 1 (f32)")
axes[1].hist(d64, bins=80, color="darkorange", edgecolor="none")
axes[1].axvline(0, color="crimson", lw=1)
axes[1].set_title("mf_loss - 1 (f64 loss on error)")
fig.tight_layout()
fig.savefig(_REPO / "outputs" / "mf_loss_probe_dev_hist.png", dpi=120)
plt.show()
print("Saved:", out_json, "and outputs/mf_loss_probe_dev_hist.png")

## Reading results

- If `abs_max` for `mf_minus_1_f32` is ~1e-4–1e-3 but UI showed `1.000`, deviation exists — **display rounding**.
- If `frac_abs_lt_1e-10` ≈ 1, loss is **saturated** near the adaptive cap (`loss_mid >> 1e-3`).
- Compare f32 vs f64: large `max_abs_diff_mf_f64_minus_f32` means float32 loss aggregation matters for tiny deviations.
- For `anisotropic_transport_coef`: target `coef × L_total` ~ 0.05–0.2 × `mf_minus_1_f32.abs_max` (or `range`).